# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dyajaballh8/FlyRank_Intern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

The feature vector uses only information observed in the May 2026 decision window.

The features focus on search visibility, search performance, traffic, engagement, and data-availability signals.

Client and content IDs are kept only as context identifiers and are not used as model features.

Missing numeric values are handled with median imputation, while missingness indicators are added so that missing data is not silently treated as zero.

No label-derived fields, future-window fields, or product-decision flags are used.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import os

# --------------------------------------------------
# 1. Connect to the FlyRank warehouse
# --------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_auth (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
)
""")

DATA_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-05/data_0.parquet"
)

# --------------------------------------------------
# 2. Load observable decision-window fields
# --------------------------------------------------

feature_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available,

    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    ga4_pageviews,
    ga4_sessions,
    ga4_users,
    ga4_engaged_sessions,
    ga4_total_engagement_sec,

    sessions_organic,
    sessions_direct,
    sessions_referral,
    sessions_social,
    sessions_paid,
    sessions_ai,

    scroll_events

FROM read_parquet('{DATA_PATH}')
WHERE report_date IS NOT NULL
LIMIT 20000
"""

raw_features = con.execute(feature_query).fetchdf()

# --------------------------------------------------
# 3. Engineer safe features
# --------------------------------------------------

features = raw_features.copy()

# Search CTR: percentage, consistent with the data dictionary
features["ctr_pct"] = (
    features["gsc_clicks"] * 100.0 /
    features["gsc_impressions"].replace(0, np.nan)
)

# Engagement rate
features["engagement_rate"] = (
    features["ga4_engaged_sessions"] * 100.0 /
    features["ga4_sessions"].replace(0, np.nan)
)

# Average engagement seconds per session
features["engagement_sec_per_session"] = (
    features["ga4_total_engagement_sec"] /
    features["ga4_sessions"].replace(0, np.nan)
)

# AI traffic share
features["ai_session_share_pct"] = (
    features["sessions_ai"] * 100.0 /
    features[
        [
            "sessions_organic",
            "sessions_direct",
            "sessions_referral",
            "sessions_social",
            "sessions_paid",
            "sessions_ai"
        ]
    ].sum(axis=1).replace(0, np.nan)
)

# Explicit missingness / availability indicators
features["gsc_available_flag"] = (
    features["gsc_data_available"].fillna(False).astype(int)
)

features["ga4_available_flag"] = (
    features["ga4_data_available"].fillna(False).astype(int)
)

features["gsc_impressions_missing"] = (
    features["gsc_impressions"].isna().astype(int)
)

features["ga4_sessions_missing"] = (
    features["ga4_sessions"].isna().astype(int)
)

# --------------------------------------------------
# 4. Select model feature columns
# --------------------------------------------------

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events",
    "ctr_pct",
    "engagement_rate",
    "engagement_sec_per_session",
    "ai_session_share_pct",
    "gsc_available_flag",
    "ga4_available_flag",
    "gsc_impressions_missing",
    "ga4_sessions_missing"
]

X = features[feature_columns].copy()

# --------------------------------------------------
# 5. Median imputation for numeric features
# --------------------------------------------------

for col in X.columns:
    if X[col].isna().any():
        X[col] = X[col].fillna(X[col].median())

# --------------------------------------------------
# 6. Final checks
# --------------------------------------------------

print("Feature vector created successfully.")
print("Rows:", len(X))
print("Features:", len(X.columns))
print("Remaining missing values:", int(X.isna().sum().sum()))

display(X.head())

assert len(X) > 0
assert X.isna().sum().sum() == 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector created successfully.
Rows: 20000
Features: 23
Remaining missing values: 0


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,...,sessions_ai,scroll_events,ctr_pct,engagement_rate,engagement_sec_per_session,ai_session_share_pct,gsc_available_flag,ga4_available_flag,gsc_impressions_missing,ga4_sessions_missing
0,0,0,14.141741,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0,0,0,0
1,0,0,14.141741,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0,0,0,0
2,0,0,14.141741,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0,0,0,0
3,0,0,14.141741,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0,0,0,0
4,0,0,14.141741,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

All model features are derived from the May 2026 decision window.

- `gsc_impressions`: search visibility volume; missing values are median-filled and missingness is separately flagged.
- `gsc_clicks`: observed search clicks; missing values are median-filled.
- `gsc_avg_position`: observed average search position; missing values are median-filled.
- `ga4_pageviews`: observed pageviews; missing values are median-filled.
- `ga4_sessions`: observed GA4 sessions; missing values are median-filled and missingness is separately flagged.
- `ga4_users`: observed GA4 users; missing values are median-filled.
- `ga4_engaged_sessions`: observed engaged sessions; missing values are median-filled.
- `ga4_total_engagement_sec`: observed engagement time; missing values are median-filled.
- `sessions_organic`: observed organic sessions; median-filled.
- `sessions_direct`: observed direct sessions; median-filled.
- `sessions_referral`: observed referral sessions; median-filled.
- `sessions_social`: observed social sessions; median-filled.
- `sessions_paid`: observed paid sessions; median-filled.
- `sessions_ai`: observed AI sessions; median-filled.
- `scroll_events`: observed scroll events; median-filled.
- `ctr_pct`: derived only from May GSC clicks and impressions; median-filled.
- `engagement_rate`: derived from May engaged sessions and sessions; median-filled.
- `engagement_sec_per_session`: derived from May engagement time and sessions; median-filled.
- `ai_session_share_pct`: derived from May traffic-channel counts; median-filled.
- `gsc_available_flag`: indicates whether GSC data is available.
- `ga4_available_flag`: indicates whether GA4 data is available.
- `gsc_impressions_missing`: explicit missingness indicator.
- `ga4_sessions_missing`: explicit missingness indicator.

`client_hash_id` and `content_hash_id` are context identifiers only and are not features.

No categorical business fields are used as predictive features. The available-data booleans are converted to explicit binary indicators.

All selected measurements are available within the decision window and do not depend on a future label.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature notes: verify missingness and data types

feature_notes = pd.DataFrame({
    "feature": feature_columns,
    "dtype": [str(X[c].dtype) for c in feature_columns],
    "missing_after_imputation": [
        int(X[c].isna().sum()) for c in feature_columns
    ]
})

display(feature_notes)

print("Total model features:", len(feature_columns))
print("Features with remaining missing values:",
      int((feature_notes["missing_after_imputation"] > 0).sum()))

assert (feature_notes["missing_after_imputation"] == 0).all()

,feature,dtype,missing_after_imputation
0,gsc_impressions,int64,0
1,gsc_clicks,int64,0
2,gsc_avg_position,float64,0
3,ga4_pageviews,int64,0
4,ga4_sessions,int64,0
5,ga4_users,int64,0
6,ga4_engaged_sessions,int64,0
7,ga4_total_engagement_sec,int64,0
8,sessions_organic,int64,0
9,sessions_direct,int64,0


Total model features: 23
Features with remaining missing values: 0


## 3. The leakage hunt

The feature vector is checked against three leakage risks:

1. Label-derived fields such as `trend_pct`, `trend_direction`, and `is_declining_label`.
2. Future-window information beyond the May 2026 decision window.
3. Product-decision or action fields.

The warehouse performance table used here contains the May 2026 snapshot. The maximum observed date must therefore remain inside May 2026.

The identifiers are also checked to ensure they are not part of the model feature vector.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------
# Leakage check 1: schema inspection
# --------------------------------------------------

schema = con.execute(f"""
DESCRIBE SELECT *
FROM read_parquet('{DATA_PATH}')
""").fetchdf()

available_columns = set(schema["column_name"].tolist())

# Known label-derived / future / decision fields that must not be used
forbidden_terms = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "label",
    "target",
    "action",
    "decision",
    "product_flag"
]

forbidden_in_features = [
    col for col in X.columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("Forbidden/label-derived columns found in feature vector:")
print(forbidden_in_features)

assert len(forbidden_in_features) == 0


# --------------------------------------------------
# Leakage check 2: future-window test
# --------------------------------------------------

window_check = con.execute(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS rows
FROM read_parquet('{DATA_PATH}')
""").fetchdf()

display(window_check)

max_date = pd.to_datetime(window_check.loc[0, "max_date"])
decision_end = pd.Timestamp("2026-05-31")

print("Decision window ends:", decision_end.date())
print("Observed maximum date:", max_date.date())

assert max_date <= decision_end


# --------------------------------------------------
# Leakage check 3: IDs are context, not features
# --------------------------------------------------

context_columns = {
    "report_date",
    "client_hash_id",
    "content_hash_id"
}

id_features = [
    col for col in X.columns
    if "client" in col.lower()
    or "content" in col.lower()
    or col in context_columns
]

print("Identifier fields inside feature vector:")
print(id_features)

assert len(id_features) == 0


# --------------------------------------------------
# Final leakage result
# --------------------------------------------------

print("\nLeakage checks PASSED.")
print("- No label-derived feature used.")
print("- No future-window feature used.")
print("- No product-decision feature used.")
print("- No client/content identifier used as a model feature.")

Forbidden/label-derived columns found in feature vector:
[]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,rows
0,2026-05-01,2026-05-31,11687376


Decision window ends: 2026-05-31
Observed maximum date: 2026-05-31
Identifier fields inside feature vector:
[]

Leakage checks PASSED.
- No label-derived feature used.
- No future-window feature used.
- No product-decision feature used.
- No client/content identifier used as a model feature.


## 4. What I excluded and why

The following fields are deliberately excluded from the feature vector:

- `report_date` — context for the decision window, not a predictive feature.
- `client_hash_id` — pseudonymous identifier used for grouping, joining, and validation; not a feature.
- `content_hash_id` — pseudonymous identifier used for identifying content; not a feature.
- `trend_pct` — excluded because it contributes to the label definition and would leak outcome information.
- `trend_direction` — excluded because it is derived from `trend_pct` and is therefore label-derived.
- `is_declining_label` — the target/label itself; never used as an input feature.
- Product-decision flags — excluded because they represent downstream decisions rather than information available to the model at prediction time.
- Future-window measurements — excluded because they would not be available at the decision moment.

The feature vector therefore contains only observable May 2026 performance and availability signals.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Explicit exclusion audit

excluded_fields = {
    "report_date": "Context / decision-window identifier",
    "client_hash_id": "Pseudonymous ID; grouping and validation only",
    "content_hash_id": "Pseudonymous ID; identification only",
    "trend_pct": "Label-derived / outcome information",
    "trend_direction": "Derived from trend_pct",
    "is_declining_label": "Target label",
    "product_decision_flags": "Downstream product decisions",
    "future_window_measurements": "Unavailable at prediction time"
}

excluded_df = pd.DataFrame(
    list(excluded_fields.items()),
    columns=["field", "reason"]
)

display(excluded_df)

print("Excluded field categories:", len(excluded_df))

# Final feature audit
print("\nFinal feature vector:")
print(feature_columns)

print("\nFinal feature count:", len(feature_columns))
print("Final missing values:", int(X.isna().sum().sum()))

assert len(feature_columns) == len(set(feature_columns))
assert X.isna().sum().sum() == 0

print("\nFeature vector and leakage/privacy audit PASSED.")

,field,reason
0,report_date,Context / decision-window identifier
1,client_hash_id,Pseudonymous ID; grouping and validation only
2,content_hash_id,Pseudonymous ID; identification only
3,trend_pct,Label-derived / outcome information
4,trend_direction,Derived from trend_pct
5,is_declining_label,Target label
6,product_decision_flags,Downstream product decisions
7,future_window_measurements,Unavailable at prediction time


Excluded field categories: 8

Final feature vector:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events', 'ctr_pct', 'engagement_rate', 'engagement_sec_per_session', 'ai_session_share_pct', 'gsc_available_flag', 'ga4_available_flag', 'gsc_impressions_missing', 'ga4_sessions_missing']

Final feature count: 23
Final missing values: 0

Feature vector and leakage/privacy audit PASSED.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.